# MandiGuard finalization checks

Colab-ready checks of committed tables. This notebook does **not** resimulate MandiGuard: `forecaster_2024.pkl` needs scikit-learn 1.3.0.

It checks H3's denominator, the Exp 1b outcome mix, the Exp 14 parity file, and the Exp 18 B1/B2 split of the already-used holdout.


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('.')
if not (ROOT / 'results').exists():
    ROOT = Path('/content')  # if files were uploaded to Colab
exp2 = pd.read_csv(ROOT / 'results/exp2_main_benchmark_6state.csv').set_index('policy')
holm = pd.read_csv(ROOT / 'results/confirmatory_holm_summary.csv').set_index('hypothesis')
h3 = float(holm.loc['H1'].name and holm.loc['H3', 'estimate'])
ep = (exp2.loc['MandiGuard', 'ep_phantom_loss_pct'] - exp2.loc['B3_MLPointDispatch', 'ep_phantom_loss_pct']) / 100
cond = (exp2.loc['MandiGuard', 'cond_phantom_loss_rate_pct'] - exp2.loc['B3_MLPointDispatch', 'cond_phantom_loss_rate_pct']) / 100
assert abs(ep - h3) < 5e-4
assert abs(cond - h3) > 0.01
print('H3 episode difference', round(ep, 4), 'conditional difference', round(cond, 4))
print('H2 status', holm.loc['H2', 'status'], 'estimate', holm.loc['H2', 'estimate'])


In [ ]:
types = pd.read_csv(ROOT / 'results/exp1b_b1_outcome_types.csv')
assert int(types['n_b1_dispatches'].sum()) == 63040
wait = types.loc[types.outcome_type.isin(['exact_wait_d1', 'exact_wait_d2']), 'share_pct'].sum()
print(types.to_string(index=False))
print('delayed exact share', round(wait, 2))
par = pd.read_csv(ROOT / 'results/exp14_official_datagov_parity_audit.csv')
print(par[['gov_dates', 'matched_panel_keys', 'modal_price_exact_parity_pct']].to_string(index=False))
sl = pd.read_csv(ROOT / 'results/exp18_used_holdout_temporal_slice.csv')
print(sl.to_string(index=False))
print('Exp 18 is a split of the used holdout. untouched is false.')
